## What Is Conversation Buffer Memory?

### The core idea in one sentence
Store every message in the conversation as a list, and re-send the entire list to the LLM on every API call.

---

### Point 1 — LLMs are stateless. They remember nothing between calls.

Every API call to an LLM is completely independent. The model does not know what was said one second ago. It only knows what is inside the current request. If you do not pass the conversation history yourself, the model has zero context of the prior exchange.

This is not a bug — it is by design. LLMs are stateless compute functions: input goes in, output comes out, nothing is stored.

---

### Point 2 — The buffer is just a list. You append to it. You re-send it.

Conversation Buffer Memory is the simplest solution to the statelessness problem. You maintain an ordered list of messages. Every user message gets appended. Every assistant response gets appended. On the next API call, you send the entire list again.

```
Turn 1 → send: [system, user_msg_1]
Turn 2 → send: [system, user_msg_1, assistant_msg_1, user_msg_2]
Turn 3 → send: [system, user_msg_1, assistant_msg_1, user_msg_2, assistant_msg_2, user_msg_3]
```

The model does not remember anything. You give it the transcript. It reads it. That is the entire mechanism.

---

### Point 3 — Each message has a role. Role order is strict.

Every message in the list has two fields — a `role` and `content`. In the OpenAI API, valid roles are:
- `system`    — the agent's instructions / persona (sent once, at the top of the list)
- `user`      — the human's message
- `assistant` — the model's response

Unlike Anthropic's API, OpenAI includes the system message **inside** the messages list as the first item, not as a separate parameter.

---

### Point 4 — Token cost grows with every turn. This is the fundamental problem.

OpenAI charges per token. A token is roughly 0.75 words. Every API call with a buffer sends all prior messages again:

```
Turn 1  → ~50 tokens sent
Turn 2  → ~130 tokens sent   (Turn 1 + Turn 2)
Turn 5  → ~450 tokens sent   (Turns 1–5 all re-sent)
Turn 10 → ~950 tokens sent   (Turns 1–10 all re-sent)
```

Per-turn cost grows linearly. Total conversation cost grows quadratically. At scale with thousands of users, this is a serious unit-economics problem.

---

### Point 5 — The context window is the hard ceiling. Hit it and the call fails.

GPT-4o supports a 128,000 token context window. This sounds large, but a long FinCoach session with tool outputs, retrieved documents, and hundreds of turns can approach it. When you exceed the context window, the API call fails — no graceful degradation, just an error.

---

## FinCoach Example — Why This Matters in Fintech

FinCoach is a personal financial advisor agent serving users in India.

**Turn 1**
> User: `"My monthly take-home is ₹1,20,000."`
> FinCoach: `"Thanks. What are your current financial goals?"`
> *API receives: system + 2 messages, ~80 tokens*

**Turn 3**
> User: `"My expenses are ₹60,000/month."`
> FinCoach: `"Your monthly surplus is ₹60,000. That's a solid base."`
> *API receives: system + 6 messages, ~200 tokens — model correctly used ₹1,20,000 from Turn 1*

**Turn 5**
> User: `"What's the most important thing I should do this month?"`
> FinCoach: *gives advice personalised to salary, expenses, and goals from Turns 1–4*
> *API receives: system + 10 messages, ~400 tokens*

The personalisation works because the entire history is in the request every time.

**Now scale this. Turn 50. 10,000 users.**
- Turn 50 sends ~8,000 tokens — 160x more than Turn 1
- A power user with 200 turns can hit 30,000+ tokens per call
- The buffer lives in RAM — a server restart means Priya opens FinCoach and the agent has no idea who she is
- A fact from Turn 3 ("I'm risk-averse") has the same weight as irrelevant small talk from Turn 48



---

## Trade-offs

| | |
|---|---|
| ✅ Perfect recall within a session | ❌ Token cost grows every turn |
| ✅ Zero implementation complexity | ❌ Hard context window ceiling |
| ✅ Deterministic — easy to debug | ❌ No persistence across sessions |
| ✅ No information loss or distortion | ❌ No prioritisation of important facts |

## Production Verdict

> **Not sufficient alone. Always present as a component.**
>
> No production system uses raw buffer memory as its only memory layer. But every production system uses *something like* a buffer for the current active session — bounded by a hard token budget, persisted to a database, and supplemented by a long-term memory layer for returning users.

---

Turn 1 API call  →  [System Prompt] + [User 1]

Turn 2 API call  →  [System Prompt] + [User 1] + [AI 1] + [User 2]

Turn 3 API call  →  [System Prompt] + [User 1] + [AI 1] + [User 2] + [AI 2] + [User 3]

Turn N API call  →  [System Prompt] + ALL previous turns + [User N]



In [13]:
# Install the two external packages this notebook needs.
# 'openai'    — the official OpenAI Python SDK for calling GPT-4o.
# 'tiktoken'  — OpenAI's own tokeniser, used to count tokens before
#               making API calls so we can enforce our token budget.
#               tiktoken is the exact tokeniser GPT-4o uses internally,
#               so token counts here are precise (not an approximation).
# '--quiet'   — suppresses installation output to keep the notebook clean.
!pip install openai tiktoken --quiet

In [3]:
# --- Standard library imports (no installation needed) ---

import json          # Converts Python dicts/lists to JSON strings for saving sessions to disk.
import os            # Reads environment variables — used to load the API key safely.
import time          # Provides sleep() to add a small delay between API calls in the demo.

from datetime import datetime, timezone  # Generates UTC timestamps; timezone needed for Python 3.12+ for every message we store.

from typing import List, Dict, Optional
# List[...]  — tells Python this variable holds a list of a specific type.
# Dict[...]  — tells Python this variable holds a dictionary with specific key/value types.
# Optional   — marks a parameter as allowed to be None.
# These are type hints — they don't change runtime behaviour, but make the code
# easier to read and help IDEs catch bugs before you run the code.

from dataclasses import dataclass, field, asdict
# dataclass  — a decorator that auto-generates __init__, __repr__, etc. for a class.
# field      — lets us set a dynamic default (like calling datetime.utcnow()) per instance.
# asdict     — converts a dataclass instance to a plain dict, needed for JSON serialisation.

# --- Third-party imports (installed above) ---

import openai        # The OpenAI SDK — provides the client for calling GPT-4o.
import tiktoken      # Token counting — lets us count tokens exactly before API calls.

In [4]:
# Read the OpenAI API key from the environment.
# os.environ.get() returns the value of the variable, or the fallback string if not set.
# NEVER paste your real API key directly in a notebook — use environment variables.
# To set it before running:
#   In terminal  : export OPENAI_API_KEY="sk-..."
#   In Colab     : paste it directly in the string below (don't commit to git)
#   In production: use AWS Secrets Manager, GCP Secret Manager, or Azure Key Vault.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "your-openai-api-key-here")

# Create the OpenAI client object.
# This is the gateway to all GPT-4o API calls in this notebook.
# It manages authentication headers, connection pooling, and retry logic.
# In production: create this once at module level and reuse it — do not recreate per request.
client = openai.OpenAI(api_key=OPENAI_API_KEY)

# The specific GPT model we will use throughout this course.
# gpt-4o: OpenAI's flagship multimodal model as of 2025.
# Supports a 128,000 token context window.
# Pricing (mid-2025): $2.50 per million input tokens, $10.00 per million output tokens.
MODEL = "gpt-4o"

# Store the context window size as a constant.
# We reference this when explaining how much of the window the buffer is consuming.
MODEL_CONTEXT_WINDOW = 128_000  # 128,000 tokens maximum per API call for gpt-4o.

# The OpenAI tokeniser for GPT-4o is called 'o200k_base'.
# We load it once here and reuse it everywhere — loading is slightly slow
# so we avoid doing it inside functions that get called repeatedly.
# Note: for gpt-4o specifically, use 'o200k_base'. For gpt-4/gpt-3.5, use 'cl100k_base'.
TOKENISER = tiktoken.get_encoding("o200k_base")

# Confirm setup is complete.
print(f"OpenAI client initialised. Model: {MODEL}")
print(f"Context window : {MODEL_CONTEXT_WINDOW:,} tokens")
print(f"Tokeniser      : o200k_base (exact tokeniser for {MODEL})")

OpenAI client initialised. Model: gpt-4o
Context window : 128,000 tokens
Tokeniser      : o200k_base (exact tokeniser for gpt-4o)


---
## Part 1 — The Message Data Model

Before building the buffer, we define the atomic unit: a single message.

**OpenAI API difference vs Anthropic:**
In the OpenAI API, the system prompt lives **inside** the messages list as a message with `role: "system"` — it is the very first item. This is different from Anthropic, where the system prompt is a separate parameter.

```python
# OpenAI format — system prompt is message[0]
[
    {"role": "system",    "content": "You are FinCoach..."},
    {"role": "user",      "content": "My salary is ₹1,20,000"},
    {"role": "assistant", "content": "Got it. How can I help?"}
]
```

We add a `timestamp` field on top of the API fields — not for the API (which ignores it), but for our own audit trail, time-based eviction, and debugging.

In [5]:
@dataclass  # Auto-generates __init__, __repr__, __eq__ — we do not need to write them manually.
class Message:
    """
    Represents a single message in the conversation.
    Stores everything we need for both API calls and production record-keeping.
    """

    role: str
    # Who sent this message.
    # OpenAI valid roles: 'system', 'user', 'assistant'.
    # 'system'    — the agent's instructions, always the first message.
    # 'user'      — the human's input.
    # 'assistant' — the model's response.
    # Passing an invalid role causes a 400 API error.

    content: str
    # The actual text of the message.
    # In production this could also be a list of content parts (text + images),
    # but we keep it as plain text here for clarity.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # When this message was created, in UTC ISO 8601 format.
    # 'default_factory' means: call this lambda each time a Message is created
    # to get a fresh timestamp — do not share one timestamp across all instances.
    # The 'Z' suffix means UTC (Zulu time) — always use UTC in production
    # to avoid timezone bugs across multi-region deployments.

    def to_api_format(self) -> Dict[str, str]:
        """
        Return only the fields the OpenAI API accepts: role and content.
        The API does not know about our 'timestamp' field and will reject it
        with a validation error. This method strips it before the API call.
        """
        return {"role": self.role, "content": self.content}
        # Returns a plain dict — exactly what client.chat.completions.create()
        # expects in its 'messages' parameter.


# --- Quick check: create one message and inspect both formats ---

sample_message = Message(role="user", content="My monthly salary is ₹1,20,000.")
# Creates a Message with role='user', the given content, and a fresh UTC timestamp.

print("Full message object (with timestamp):")
print(sample_message)
# Shows all three fields including the auto-generated timestamp.

print("\nAPI format (timestamp stripped):")
print(sample_message.to_api_format())
# Shows only role and content — what the OpenAI API actually receives.

Full message object (with timestamp):
Message(role='user', content='My monthly salary is ₹1,20,000.', timestamp='2026-06-12T03:47:28.502397+00:00')

API format (timestamp stripped):
{'role': 'user', 'content': 'My monthly salary is ₹1,20,000.'}


---
## Part 2 — The ConversationBufferMemory Class

**Key OpenAI difference:** The system prompt lives inside the messages list here. The buffer's `get_messages_for_api()` method always prepends the system message as the first item before returning the list to the API.

Everything else — token budgeting, session scoping, persistence, overflow handling — is identical in concept to any other LLM backend.

In [6]:
class ConversationBufferMemory:
    """
    Holds the full conversation history and re-sends it on every OpenAI API call.
    Adds a hard token budget, session scoping, and persistence on top of
    the naive 'list of dicts' pattern.

    OpenAI-specific note: the system prompt is stored separately but injected
    as the first message in get_messages_for_api() — this is the OpenAI convention.
    """

    # ------------------------------------------------------------------
    # INITIALISATION
    # ------------------------------------------------------------------

    def __init__(
        self,
        session_id: str,               # Unique ID for this conversation session.
        user_id: str,                  # ID of the user who owns this session.
        system_prompt: str,            # The agent's instructions (FinCoach's persona).
        max_token_budget: int = 4_000, # Hard cap: buffer conversation messages may not exceed this.
    ):
        self.session_id = session_id
        # Stored on the object so every method can reference which session this is.
        # Used in log lines, persistence keys, and access control checks.

        self.user_id = user_id
        # Critical for multi-tenant isolation.
        # In production: validate this against your auth system before
        # granting any read or write access to the buffer.

        self.system_prompt = system_prompt
        # The agent's instruction set — FinCoach's persona and rules.
        # Stored separately from self.messages because we always want it
        # at position [0] in the API payload, not mixed into the conversation.
        # get_messages_for_api() handles injecting it correctly.

        self.max_token_budget = max_token_budget
        # The single most important production addition.
        # This cap applies to the CONVERSATION messages only (not the system prompt).
        # Without it, a long session silently grows until the API call
        # either fails (context overflow) or costs 100x what you expected.
        # Default of 4,000 leaves ample room in GPT-4o's 128k context window
        # for the system prompt, retrieved memories, and the model's response.

        self.messages: List[Message] = []
        # The conversation history — an ordered list of Message objects.
        # Does NOT include the system prompt (that is in self.system_prompt).
        # Starts empty. Grows by two messages per turn (one user, one assistant).
        # Order must never be changed — the LLM uses sequence to understand context.

        self._tokeniser = TOKENISER
        # Reference the module-level tokeniser loaded at setup.
        # We use the module-level one to avoid reloading it per buffer instance.
        # The leading underscore signals this is internal — callers should not
        # access it directly.

        self.created_at = datetime.now(timezone.utc).isoformat()
        # Timestamp of when this buffer was created.
        # Stored in the session record for audit trail and analytics.

        print(f"[Buffer] Initialised — session: {self.session_id}, "
              f"user: {self.user_id}, token budget: {self.max_token_budget:,}")
        # Confirmation so it's clear when a new session starts in the notebook output.

    # ------------------------------------------------------------------
    # TOKEN COUNTING
    # ------------------------------------------------------------------

    def _count_tokens(self, text: str) -> int:
        """
        Count the exact number of tokens in a string using GPT-4o's tokeniser.
        Unlike with Anthropic (where tiktoken is an approximation),
        here tiktoken IS the exact tokeniser GPT-4o uses — counts are precise.
        """
        return len(self._tokeniser.encode(text))
        # encode() converts the string into a list of integer token IDs.
        # len() of that list is the exact token count.
        # Example: "Hello world" → [9906, 1917] → 2 tokens.

    def get_total_tokens(self) -> int:
        """
        Sum the tokens across all CONVERSATION messages in the buffer.
        Does NOT include the system prompt tokens — those are a separate overhead.
        This is your primary cost-monitoring metric per session.
        """
        return sum(self._count_tokens(msg.content) for msg in self.messages)
        # sum(...) adds up the token count of every message's content field.
        # Role labels ('user', 'assistant') add ~3–4 tokens per message overhead
        # in the OpenAI format, but we ignore that here — it's negligible for budgeting.

    def get_system_prompt_tokens(self) -> int:
        """
        Count tokens in the system prompt separately.
        Useful for understanding the full API call cost:
        total input tokens = system prompt tokens + conversation tokens.
        """
        return self._count_tokens(self.system_prompt)
        # The system prompt is constant per session — its token count
        # adds a fixed overhead to every single API call.

    def get_budget_utilisation(self) -> float:
        """
        Return conversation token usage as a fraction of the budget.
        0.0 = empty buffer. 1.0 = exactly at budget. >1.0 = over budget.
        Alert when this exceeds 0.8 — gives you time to act before hitting the hard limit.
        """
        return self.get_total_tokens() / self.max_token_budget
        # Simple division: current conversation tokens ÷ maximum allowed.

    # ------------------------------------------------------------------
    # CORE BUFFER OPERATIONS
    # ------------------------------------------------------------------

    def add_message(self, role: str, content: str) -> None:
        """
        Append one message to the conversation buffer.
        Validates the role and checks the token budget before appending.
        Raises BufferOverflowError if the budget would be exceeded.
        Note: use role 'user' or 'assistant' only — the system prompt
        is managed separately via self.system_prompt.
        """

        if role not in ("user", "assistant"):
            # Guard against passing 'system' here — system prompt is separate.
            # Any other invalid role would also be caught by this check.
            raise ValueError(
                f"Invalid role '{role}'. Use 'user' or 'assistant'. "
                f"The system prompt is managed via the system_prompt attribute."
            )

        new_message = Message(role=role, content=content)
        # Create the Message object. The timestamp is auto-set to UTC now.

        new_message_tokens = self._count_tokens(content)
        # Count how many tokens this new message would add to the buffer.

        current_total = self.get_total_tokens()
        # How many tokens are already in the conversation buffer right now.

        projected_total = current_total + new_message_tokens
        # What the buffer size would be AFTER adding this message.

        if projected_total > self.max_token_budget:
            # The budget would be exceeded — raise an explicit, named error.
            # We check BEFORE appending — never after.
            # Explicit error >> silent truncation: the caller knows exactly
            # what happened and can apply the correct overflow policy.
            raise BufferOverflowError(
                f"Adding this message ({new_message_tokens} tokens) would bring "
                f"the buffer to {projected_total} tokens, exceeding the budget of "
                f"{self.max_token_budget}. Current usage: {current_total} tokens."
            )

        self.messages.append(new_message)
        # All checks passed — append the message to the end of the list.
        # append() adds to the end, preserving the chronological order the model needs.

    def get_messages_for_api(self) -> List[Dict[str, str]]:
        """
        Return the complete message list in the exact format the OpenAI API expects.

        IMPORTANT — OpenAI convention:
        The system prompt must be the FIRST item in the messages list.
        This method prepends it automatically so the caller never has to think about it.

        Returns:
            [
                {"role": "system",    "content": "<system prompt>"},   ← always first
                {"role": "user",      "content": "<user message 1>"},
                {"role": "assistant", "content": "<response 1>"},
                {"role": "user",      "content": "<user message 2>"},
                ...
            ]
        """
        system_message = {"role": "system", "content": self.system_prompt}
        # Build the system message dict in the OpenAI API format.
        # This is prepended to every API call — it is always the first message.

        conversation = [msg.to_api_format() for msg in self.messages]
        # Convert each Message object in the buffer to a plain dict.
        # to_api_format() strips the timestamp and returns {role, content}.

        return [system_message] + conversation
        # Concatenate: system message first, then the conversation history.
        # The [system_message] wraps the dict in a list so + works correctly.

    def clear(self) -> None:
        """
        Empty the conversation buffer completely.
        ALWAYS call persist() before clear() — never lose a session without saving.
        The system prompt is NOT cleared — it stays on the object.
        """
        self.messages = []
        # Replace the list with a new empty list.
        # Python garbage-collects the old Message objects automatically.

    def turn_count(self) -> int:
        """
        Return the number of complete conversation turns.
        One turn = one user message + one assistant response = 2 messages.
        Use this to trigger summarisation policies (e.g., 'summarise after 20 turns').
        """
        return len(self.messages) // 2
        # Integer division: 6 messages ÷ 2 = 3 complete turns.

    # ------------------------------------------------------------------
    # PERSISTENCE — SAVE AND LOAD
    # ------------------------------------------------------------------

    def persist(self, filepath: str) -> None:
        """
        Save the full buffer state to a JSON file.
        In production: write to PostgreSQL, Firestore, or DynamoDB instead.
        The schema here is database-ready — the same fields map cleanly to DB columns.
        """

        session_record = {
            "schema_version": "1.0",
            # Version the schema. When you change its structure, bump this number
            # and write a migration function for records saved with old versions.

            "session_id": self.session_id,
            # The unique identifier for this session — primary key in production.

            "user_id": self.user_id,
            # The user who owns this session — used for access control on load.

            "model": MODEL,
            # Record which model was used — important if you switch models later
            # and need to know which tokeniser applied to this session's token counts.

            "created_at": self.created_at,
            # When this session started — for audit trail and session analytics.

            "saved_at": datetime.now(timezone.utc).isoformat(),
            # When this snapshot was written — helps detect stale or duplicate saves.

            "token_stats": {
                "conversation_tokens": self.get_total_tokens(),
                # Conversation token count at save time — for cost analysis.
                "system_prompt_tokens": self.get_system_prompt_tokens(),
                # System prompt tokens — fixed overhead per API call, logged for clarity.
                "max_token_budget": self.max_token_budget,
                # Save the budget so it is restored correctly on load.
                "budget_utilisation": round(self.get_budget_utilisation(), 4),
                # How full the buffer was — helps decide if summarisation is needed next session.
                "turn_count": self.turn_count(),
                # How many turns happened — for session-level analytics.
            },

            "messages": [asdict(msg) for msg in self.messages],
            # asdict() converts each Message dataclass to a plain dict
            # INCLUDING the timestamp field — full fidelity for audit purposes.
            # The list comprehension applies it to every message in the buffer.
            # Note: system prompt is NOT in this list — it is stored separately.
        }

        with open(filepath, "w", encoding="utf-8") as f:
            # Open the file for writing ('w') with UTF-8 encoding.
            # UTF-8 is required because financial data often contains
            # non-ASCII characters like ₹ (the Indian Rupee symbol).
            json.dump(session_record, f, indent=2, ensure_ascii=False)
            # json.dump() serialises the dict to a JSON string and writes it to the file.
            # indent=2    : pretty-prints with 2-space indentation — human-readable for debugging.
            # ensure_ascii=False: writes ₹ as-is instead of \u20b9 unicode escape.

        print(f"[Buffer] Persisted to '{filepath}' — "
              f"{self.turn_count()} turns, {self.get_total_tokens()} conversation tokens")

    @classmethod
    def load(cls, filepath: str) -> "ConversationBufferMemory":
        """
        Restore a buffer from a previously saved JSON file.
        Use this to resume a session for a returning user.
        @classmethod means we call it on the class: ConversationBufferMemory.load(...)
        """

        with open(filepath, "r", encoding="utf-8") as f:
            session_record = json.load(f)
            # json.load() reads the file and parses it back into a Python dict.

        if session_record.get("schema_version") != "1.0":
            # Reject files with an unknown schema version rather than
            # silently loading data in an incompatible format.
            # In production: route to the correct migration function.
            raise ValueError(
                f"Unknown schema version: {session_record.get('schema_version')}. "
                f"Expected '1.0'."
            )

        buffer = cls(
            # cls(...) calls __init__ on the class — equivalent to ConversationBufferMemory(...).
            session_id=session_record["session_id"],
            # Restore the original session ID, not generate a new one.
            user_id=session_record["user_id"],
            # Restore the user ID — validate against logged-in user in production.
            system_prompt="[LOADED FROM FILE — set system_prompt after loading]",
            # The system prompt is agent config, not user data — it is not stored in the file.
            # The caller must set it after loading: restored_buffer.system_prompt = ...
            max_token_budget=session_record["token_stats"]["max_token_budget"],
            # Restore the exact same token budget that was active when the session was saved.
        )

        buffer.messages = [
            Message(
                role=msg_dict["role"],            # Restore the speaker role.
                content=msg_dict["content"],      # Restore the message text.
                timestamp=msg_dict["timestamp"],  # Restore the original timestamp.
            )
            for msg_dict in session_record["messages"]
            # Iterate over every saved message dict and reconstruct a Message object.
        ]

        buffer.created_at = session_record["created_at"]
        # Restore the original session start time — do not reset it to now.

        print(f"[Buffer] Loaded from '{filepath}' — "
              f"{buffer.turn_count()} turns, {buffer.get_total_tokens()} tokens")

        return buffer
        # Return the fully reconstructed buffer, ready to resume the conversation.

    # ------------------------------------------------------------------
    # DIAGNOSTICS
    # ------------------------------------------------------------------

    def print_stats(self) -> None:
        """
        Print a human-readable summary of buffer state.
        In production: emit these as structured metrics to your observability stack
        (Prometheus, CloudWatch, Datadog) rather than printing them.
        """
        total_tokens    = self.get_total_tokens()        # Current conversation token count.
        system_tokens   = self.get_system_prompt_tokens()# System prompt overhead (constant).
        utilisation     = self.get_budget_utilisation()  # Fraction of budget used.

        print(f"\n{'='*58}")
        print(f"  Buffer Stats — Session: {self.session_id}")
        print(f"{'='*58}")
        print(f"  User ID              : {self.user_id}")
        print(f"  Model                : {MODEL}")
        print(f"  Turn count           : {self.turn_count()}")
        print(f"  Message count        : {len(self.messages)}")
        # len() on the list gives total individual messages, not turns.
        print(f"  Conversation tokens  : {total_tokens:,}")
        # :, formats the number with comma separators: 4000 → 4,000.
        print(f"  System prompt tokens : {system_tokens:,}  (constant overhead per call)")
        print(f"  Total per-call input : ~{total_tokens + system_tokens:,} tokens")
        # What each API call costs in input tokens right now.
        print(f"  Token budget         : {self.max_token_budget:,}")
        print(f"  Budget utilisation   : {utilisation:.1%}")
        # :.1% formats as a percentage with one decimal place: 0.734 → 73.4%.

        bar_length = 30
        # The visual progress bar will be 30 characters wide.
        filled = int(bar_length * min(utilisation, 1.0))
        # min() caps at 1.0 so the bar never overflows its own width.
        # int() converts the float to a whole number of filled characters.
        bar = "█" * filled + "░" * (bar_length - filled)
        # Build the bar string: filled blocks + empty blocks to total 30 characters.
        print(f"  [{bar}] {utilisation:.1%}")
        print(f"{'='*58}\n")


class BufferOverflowError(Exception):
    """
    Raised when a new message would push the buffer over its token budget.

    This is intentionally an explicit, named exception — not a silent truncation.
    The code that calls add_message() must handle it with a defined policy:
      - Summarise old messages first            → Technique 03
      - Evict the oldest messages               → Technique 02
      - Archive the session and start fresh     → Technique 04
      - Alert ops that this session is too long → Observability layer

    Silent truncation would be worse: the model would answer with incomplete
    context and neither the agent nor the caller would know.
    """
    pass
    # 'pass' because this exception needs no extra behaviour —
    # it inherits everything it needs from the base Exception class.


print("ConversationBufferMemory and BufferOverflowError defined.")

ConversationBufferMemory and BufferOverflowError defined.


---
## Part 3 — The FinCoach Agent

Wire the buffer to GPT-4o to create a working financial advisor agent.

**OpenAI API call:** `client.chat.completions.create()` — note this is different from Anthropic's `client.messages.create()`.

In [7]:
# FinCoach's system prompt — defines the agent's persona and behaviour rules.
# In production: stored in a versioned config store, not hardcoded here.
# Different user tiers (free vs premium) may receive different system prompts.
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Always personalise advice using information the user has shared in this conversation.
- Be specific with numbers when the user has provided their financial details.
- Flag when you are making assumptions due to missing information.
- Keep responses concise — 3 to 5 sentences unless the user asks for detail.
- Never provide specific buy/sell recommendations on individual stocks.
- Always recommend consulting a SEBI-registered advisor for major financial decisions.

Use all context in the conversation history to provide personalised, consistent advice."""


def chat(
    buffer: ConversationBufferMemory,  # The session's buffer — holds the full history.
    user_message: str,                  # The user's current message text.
    verbose: bool = True                # If True, print token stats after each turn.
) -> str:
    """
    Send one user message to FinCoach via GPT-4o. Return the assistant's reply.

    The four steps:
    1. Add the user message to the buffer.
    2. Send the full buffer (system + conversation) to GPT-4o.
    3. Extract the response text.
    4. Add the response to the buffer.
    """

    # STEP 1 — Add the user's message to the buffer.
    buffer.add_message(role="user", content=user_message)
    # This may raise BufferOverflowError if the budget is exceeded.
    # In production: wrap in try/except and apply your overflow policy here.

    # STEP 2 — Call the OpenAI Chat Completions API with the full history.
    response = client.chat.completions.create(
        model=MODEL,
        # Which GPT model to use — defined as 'gpt-4o' at the top of the notebook.

        max_tokens=1024,
        # Maximum number of tokens in the model's response.
        # 1024 is sufficient for most financial advice responses.
        # In production: tune this per use case — short-answer queries
        # don't need a 4096-token budget.

        messages=buffer.get_messages_for_api(),
        # THE BUFFER IN ACTION.
        # get_messages_for_api() returns:
        #   [{"role": "system", "content": "..."},   ← FinCoach's instructions
        #    {"role": "user",   "content": "..."},   ← Turn 1 user
        #    {"role": "assistant", "content": "..."}, ← Turn 1 response
        #    ...all prior turns...
        #    {"role": "user",   "content": "..."}]   ← Current user message
        # GPT-4o reads this full transcript and generates the next response.
        # NOTE: In OpenAI's API, there is no separate 'system' parameter —
        # it is all in this single 'messages' list. This differs from Anthropic.

        temperature=0.7,
        # Controls randomness in the model's output.
        # 0.0 = fully deterministic (same input always gives same output).
        # 1.0 = more creative and varied responses.
        # 0.7 is a good balance for financial advice — personalised but not hallucination-prone.
        # In production: lower temperature (0.3–0.5) for advice that must be consistent.
    )

    # STEP 3 — Extract the assistant's reply text from the OpenAI response object.
    assistant_reply = response.choices[0].message.content
    # response.choices  : a list of possible completions (usually just one).
    # [0]               : the first (and default) completion.
    # .message          : the full message object for this completion.
    # .content          : the actual text string of the assistant's reply.
    # OpenAI response structure differs from Anthropic:
    #   OpenAI  → response.choices[0].message.content
    #   Anthropic → response.content[0].text

    # STEP 4 — Add the assistant's reply to the buffer.
    buffer.add_message(role="assistant", content=assistant_reply)
    # Now the buffer contains: [...previous turns..., user_msg_N, assistant_reply_N].
    # This completed turn will be sent again on the NEXT API call.

    if verbose:
        # response.usage contains the exact token counts billed by OpenAI.
        # prompt_tokens     : all tokens sent (system + conversation history + current message).
        # completion_tokens : tokens in the assistant's response.
        # total_tokens      : prompt_tokens + completion_tokens.
        print(f"[Turn {buffer.turn_count()}] "
              f"API tokens — prompt: {response.usage.prompt_tokens}, "
              f"completion: {response.usage.completion_tokens}, "
              f"total: {response.usage.total_tokens} | "
              f"Buffer: {buffer.get_total_tokens()} conv tokens "
              f"({buffer.get_budget_utilisation():.1%} of budget)")
        # Watch how 'prompt' tokens grow with each turn — that is buffer cost compounding.

    return assistant_reply
    # Return the plain text string so the caller can display it to the user.


print("FinCoach system prompt and chat() function defined.")

FinCoach system prompt and chat() function defined.


---
## Part 4 — Live Demo: 5-Turn FinCoach Conversation

Watch two things as this runs:
1. **The agent uses earlier facts in later answers** — salary from Turn 1 informs advice in Turn 5.
2. **The `prompt` token count grows with every turn** — that is the buffer cost compounding in real time.

In [8]:
# Create a new FinCoach buffer for user 'chiru_001', session 'demo_001'.
fincoach_buffer = ConversationBufferMemory(
    session_id="session_demo_001",         # Readable ID for this demo — use UUID4 in production.
    user_id="chiru_001",                   # The user this session belongs to.
    system_prompt=FINCOACH_SYSTEM_PROMPT,  # FinCoach's instructions.
    max_token_budget=4_000,                # Buffer may use at most 4,000 conversation tokens.
)

# Five conversation turns designed to test cross-turn memory.
# Each later message deliberately references information from earlier turns.
demo_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    # Turn 1: introduces salary — the model should remember this for Turn 5.
    "I have monthly expenses of about ₹60,000 — rent, groceries, transport, and utilities.",
    # Turn 2: introduces expenses — model can now compute ₹60,000 monthly surplus.
    "I currently have no investments except a small FD of ₹50,000 that matures in 3 months.",
    # Turn 3: introduces the only existing asset.
    "What should I do with the FD maturity amount? I'm a bit risk-averse.",
    # Turn 4: first real advice question — model must use salary, expenses, and FD amount.
    "Based on everything I've told you, what's the most important financial action I should take this month?",
    # Turn 5: synthesis question — model must pull together all prior context.
]

print("Starting FinCoach demo conversation...")
print("=" * 65)

per_turn_tokens = []
# Track buffer token count after each turn to observe the growth.

for i, user_message in enumerate(demo_turns, start=1):
    # enumerate(demo_turns, start=1) yields (1, turn1), (2, turn2), ...
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")

    response = chat(buffer=fincoach_buffer, user_message=user_message, verbose=True)
    # Send this turn to GPT-4o via the buffer and get FinCoach's reply.
    # verbose=True prints the token usage breakdown after each call.

    print(f"FinCoach: {response}")

    per_turn_tokens.append(fincoach_buffer.get_total_tokens())
    # Record the conversation buffer size after this turn — for growth analysis below.

    time.sleep(1)
    # 1-second pause to avoid hitting OpenAI's rate limits during the demo.

print("\n" + "=" * 65)
print("Demo complete.")
fincoach_buffer.print_stats()
# Show the final buffer state: turns, messages, token counts, budget bar.

[Buffer] Initialised — session: session_demo_001, user: chiru_001, token budget: 4,000
Starting FinCoach demo conversation...

--- Turn 1 ---
User: Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.
[Turn 1] API tokens — prompt: 162, completion: 117, total: 279 | Buffer: 136 conv tokens (3.4% of budget)
FinCoach: Hi Chiru! With a monthly take-home salary of ₹1,20,000, you have a good foundation to start planning your finances. To get started, you might consider allocating around 50% of your income to essential expenses like housing, utilities, and groceries. Then, aim to save at least 20% of your income, which would be ₹24,000, for short-term and long-term goals. The remaining 30% can be used for discretionary spending. Does this allocation align with your current lifestyle, or do you have specific financial goals you're working towards?

--- Turn 2 ---
User: I have monthly expenses of about ₹60,000 — rent, groceries, transport, and utilities.
[Turn 2] API tokens — prompt: 307, c

---
## Part 5 — Token Growth Analysis (No API Calls)

This simulates what happens to token costs over 10 turns — no real API calls needed.
**Run this before deploying any buffer-based system.** Know the cost curve before it shows up on your OpenAI bill.

In [9]:
def analyse_token_growth(turns: List[str], avg_response_tokens: int = 80) -> None:
    """
    Simulate token growth and API cost over a multi-turn conversation.
    No API calls made — pure token arithmetic using tiktoken.

    Args:
        turns:                List of simulated user message strings.
        avg_response_tokens:  Estimated average token length of GPT-4o's responses.
    """

    INPUT_COST_PER_TOKEN  = 2.50  / 1_000_000
    # GPT-4o input cost: $2.50 per million tokens (mid-2025 pricing).
    OUTPUT_COST_PER_TOKEN = 10.00 / 1_000_000
    # GPT-4o output cost: $10.00 per million tokens (mid-2025 pricing).
    # Output is 4x more expensive than input — keep max_tokens tight in production.

    system_prompt_tokens = len(TOKENISER.encode(FINCOACH_SYSTEM_PROMPT))
    # Count system prompt tokens once — sent as the first message on EVERY API call.
    # This is a constant overhead added to every turn's input cost.

    cumulative_history_tokens = 0
    # Running total of CONVERSATION tokens (not including system prompt).
    # Grows with every turn as more messages are added to the buffer.

    total_billed_tokens = 0
    # Grand total of all tokens billed across the entire conversation so far.

    # Print the table header.
    print(f"System prompt overhead : {system_prompt_tokens} tokens (constant per call)\n")
    print(f"{'Turn':>5} | {'Msg Tokens':>10} | {'Conv History':>12} | "
          f"{'Prompt to API':>13} | {'Cumul. Cost':>11}")
    print("-" * 65)

    for i, user_msg in enumerate(turns, start=1):

        user_msg_tokens = len(TOKENISER.encode(user_msg))
        # Exact token count of THIS turn's user message, using GPT-4o's tokeniser.

        if i == 1:
            cumulative_history_tokens += user_msg_tokens
            # Turn 1: only the user message exists — no prior assistant response yet.
        else:
            cumulative_history_tokens += user_msg_tokens + avg_response_tokens
            # Turns 2+: add this user message PLUS the previous assistant response.
            # Both were added to the buffer after the previous turn and will be re-sent.

        input_tokens_this_call = system_prompt_tokens + cumulative_history_tokens
        # Total prompt tokens billed for THIS API call:
        # constant system prompt overhead + growing conversation history.

        total_billed_tokens += input_tokens_this_call + avg_response_tokens
        # Add this call's prompt and completion tokens to the running grand total.

        total_cost = (
            total_billed_tokens * INPUT_COST_PER_TOKEN +
            (i * avg_response_tokens) * (OUTPUT_COST_PER_TOKEN - INPUT_COST_PER_TOKEN)
        )
        # Cumulative cost in USD across all turns so far.
        # The second term accounts for the premium rate on output tokens.

        print(f"{i:>5} | {user_msg_tokens:>10,} | {cumulative_history_tokens:>12,} | "
              f"{input_tokens_this_call:>13,} | ${total_cost:>10.6f}")
        # :>5  = right-align in 5 characters.
        # :,   = add comma thousands separators.
        # .6f  = 6 decimal places for the cost — small numbers need precision.

    # Summary row.
    print("-" * 65)
    first_turn_prompt = system_prompt_tokens + len(TOKENISER.encode(turns[0]))
    last_turn_prompt  = system_prompt_tokens + cumulative_history_tokens
    growth_factor     = last_turn_prompt / first_turn_prompt
    # How many times more expensive is the last turn's API call vs the first?

    print(f"\nKey observation:")
    print(f"  Turn 1  prompt to API  : {first_turn_prompt:,} tokens")
    print(f"  Turn {len(turns)} prompt to API  : {last_turn_prompt:,} tokens")
    print(f"  Growth factor          : {growth_factor:.1f}x")
    print(f"\nTurn {len(turns)} costs {growth_factor:.0f}x more than Turn 1 in input tokens.")
    print(f"This is the buffer cost problem. Every other technique in this course exists to fix it.")


# Simulate a realistic 10-turn FinCoach session.
simulated_turns = [
    "Hi, I'm Chiru. My monthly salary is ₹1,20,000.",
    "My monthly expenses are ₹60,000.",
    "I have an FD of ₹50,000 maturing in 3 months.",
    "I'm risk-averse and prefer stable returns.",
    "Should I invest in mutual funds or fixed deposits?",
    "What about SIPs? How much should I invest monthly?",
    "Is ₹10,000 per month in SIPs realistic given my expenses?",
    "Which fund categories would you recommend for a conservative investor?",
    "How long should I stay invested to see meaningful returns?",
    "Can you give me a complete monthly budget breakdown based on my salary?",
]
# Each of these turns and its GPT-4o response get re-sent cumulatively on every API call.

analyse_token_growth(simulated_turns)

System prompt overhead : 132 tokens (constant per call)

 Turn | Msg Tokens | Conv History | Prompt to API | Cumul. Cost
-----------------------------------------------------------------
    1 |         17 |           17 |           149 | $  0.001173
    2 |          9 |          106 |           238 | $  0.002568
    3 |         16 |          202 |           334 | $  0.004203
    4 |          9 |          291 |           423 | $  0.006060
    5 |         10 |          381 |           513 | $  0.008143
    6 |         12 |          473 |           605 | $  0.010455
    7 |         15 |          568 |           700 | $  0.013005
    8 |         11 |          659 |           791 | $  0.015783
    9 |         11 |          750 |           882 | $  0.018788
   10 |         14 |          844 |           976 | $  0.022028
-----------------------------------------------------------------

Key observation:
  Turn 1  prompt to API  : 149 tokens
  Turn 10 prompt to API  : 976 tokens
  Growth fact

---
## Part 6 — Session Persistence: Simulating a Returning User

In [10]:
SESSION_FILE = "/tmp/fincoach_openai_session_demo_001.json"
# Path where the session JSON will be saved.
# In production: replace with a database write keyed on (user_id, session_id).

fincoach_buffer.persist(SESSION_FILE)
# Save the buffer we built in Part 4 to disk.
# This represents what happens at the end of a user's session.

print("\n--- Simulating service restart ---\n")
# In production: this is a pod restart, a new Lambda invocation,
# or the user opening the app on a different device.
# The in-memory buffer object is gone. Only the persisted file remains.

restored_buffer = ConversationBufferMemory.load(SESSION_FILE)
# Reload the buffer from the file — reconstructs the full Message list.

restored_buffer.system_prompt = FINCOACH_SYSTEM_PROMPT
# The system prompt is agent config, not user data — it is not stored in the session file.
# We must set it manually after loading before making any API calls.

print("\nAsking a follow-up that requires memory of the prior session:")

follow_up = "Quick question — based on my salary and expenses from before, what's my monthly surplus?"
# This question can ONLY be answered correctly if the prior session was restored.
# Salary (₹1,20,000) and expenses (₹60,000) were both shared in the earlier session.
print(f"User: {follow_up}")

response = chat(buffer=restored_buffer, user_message=follow_up, verbose=True)
# Send the question with the restored conversation history.
# If restoration worked: FinCoach answers ₹60,000 surplus without being told again.
# If restoration failed: FinCoach asks for the salary and expenses again.

print(f"FinCoach: {response}")
print("\nIf FinCoach correctly said ₹60,000 without being re-told — persistence worked.")

[Buffer] Persisted to '/tmp/fincoach_openai_session_demo_001.json' — 5 turns, 732 conversation tokens

--- Simulating service restart ---

[Buffer] Initialised — session: session_demo_001, user: chiru_001, token budget: 4,000
[Buffer] Loaded from '/tmp/fincoach_openai_session_demo_001.json' — 5 turns, 732 tokens

Asking a follow-up that requires memory of the prior session:
User: Quick question — based on my salary and expenses from before, what's my monthly surplus?
[Turn 6] API tokens — prompt: 932, completion: 54, total: 986 | Buffer: 803 conv tokens (20.1% of budget)
FinCoach: Based on your monthly take-home salary of ₹1,20,000 and your expenses of ₹60,000, your monthly surplus is ₹60,000. This surplus can be allocated towards savings, investments, and discretionary spending, depending on your financial goals and priorities.

If FinCoach correctly said ₹60,000 without being re-told — persistence worked.


---
## Part 7 — BufferOverflowError: The Production Safety Net

In [11]:
# Create a buffer with a tiny token budget to force an overflow quickly.
# In production: budget would be 4,000–8,000 tokens.
# Here: 100 tokens so we can trigger the overflow in just a few messages
# without waiting for a long conversation.
tiny_buffer = ConversationBufferMemory(
    session_id="session_overflow_demo",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_token_budget=100,  # Artificially small — will overflow after ~3–4 messages.
)

print("Adding messages until the budget is exceeded...\n")

messages_to_add = [
    ("user",      "My salary is ₹1,20,000 per month."),
    # ~10 tokens
    ("assistant", "Understood. Your monthly take-home is ₹1,20,000. How can I help?"),
    # ~18 tokens
    ("user",      "My monthly expenses are ₹60,000 covering rent, food, and transport."),
    # ~17 tokens — running total ~45 tokens
    ("assistant", "Got it. That gives you a monthly surplus of ₹60,000. Solid base."),
    # ~18 tokens — running total ~63 tokens
    ("user",      "I am wondering whether I should start a SIP or build an emergency fund first given my current financial situation."),
    # ~25 tokens — projected total ~88 tokens — may still fit
    ("user",      "Also what is the ideal emergency fund size for someone with my income and expense profile in India today?"),
    # ~25 tokens — projected total ~113 tokens — should trigger overflow
]

for role, content in messages_to_add:
    # Unpack each tuple into role and content.
    try:
        tiny_buffer.add_message(role=role, content=content)
        # Attempt to add the message.
        # If the budget is not exceeded, this succeeds and we print confirmation.
        print(f"  Added [{role:>9}]: '{content[:50]}...' "
              f"| Buffer: {tiny_buffer.get_total_tokens()} tokens")
        # :>9 right-aligns the role label in 9 characters for visual alignment in output.

    except BufferOverflowError as e:
        # Budget exceeded — the explicit error is raised and caught here.
        # We handle it gracefully rather than letting it crash the agent.
        print(f"\n  [OVERFLOW CAUGHT]")
        print(f"  {e}")
        print("\n  Production response options:")
        print("    1. Summarise the oldest messages to free budget  →  Technique 03")
        print("    2. Evict the oldest messages entirely            →  Technique 02")
        print("    3. Archive session and start fresh with summary  →  Technique 04")
        print("    4. Alert ops — this session is unusually long")
        break
        # Stop processing after the first overflow for this demo.
        # In production: apply the policy above, then retry the failed add_message().

[Buffer] Initialised — session: session_overflow_demo, user: chiru_001, token budget: 100
Adding messages until the budget is exceeded...

  Added [     user]: 'My salary is ₹1,20,000 per month....' | Buffer: 12 tokens
  Added [assistant]: 'Understood. Your monthly take-home is ₹1,20,000. H...' | Buffer: 32 tokens
  Added [     user]: 'My monthly expenses are ₹60,000 covering rent, foo...' | Buffer: 48 tokens
  Added [assistant]: 'Got it. That gives you a monthly surplus of ₹60,00...' | Buffer: 66 tokens
  Added [     user]: 'I am wondering whether I should start a SIP or bui...' | Buffer: 87 tokens

  [OVERFLOW CAUGHT]
  Adding this message (20 tokens) would bring the buffer to 107 tokens, exceeding the budget of 100. Current usage: 87 tokens.

  Production response options:
    1. Summarise the oldest messages to free budget  →  Technique 03
    2. Evict the oldest messages entirely            →  Technique 02
    3. Archive session and start fresh with summary  →  Technique 04
    4.

Why this matters for FinCoach specifically ?

A user tells FinCoach their salary in Turn 1. 

In Turn 50 they ask about investment options. FinCoach remembers the salary perfectly — because the Turn 1 message is still sitting right there in the list being sent on every call.

But that Turn 50 call just sent 50 turns of conversation to OpenAI. At scale with 10,000 users all having 50-turn sessions — that token bill is enormous.

That one problem — perfect memory but explosive cost — is the reason every single technique from Technique 02 to Technique 30 exists. Each one is an answer to the question: how do we keep the memory without paying the full price?

---
## Key Takeaways

**What you built:** A production-aware `ConversationBufferMemory` class using the OpenAI GPT-4o API, with hard token budget enforcement, session scoping, JSON persistence, and explicit overflow handling — anchored to the FinCoach financial advisor scenario.

**OpenAI vs Anthropic — the one structural difference:**

| | OpenAI | Anthropic |
|---|---|---|
| System prompt location | First item inside `messages` list | Separate `system` parameter |
| API method | `client.chat.completions.create()` | `client.messages.create()` |
| Response text | `response.choices[0].message.content` | `response.content[0].text` |
| Tokeniser | `tiktoken` — exact match | `tiktoken` — approximation |
| GPT-4o tokeniser | `o200k_base` | `cl100k_base` |

Everything else — the buffer logic, token budgeting, persistence schema, overflow handling — is identical regardless of which LLM provider you use. The memory architecture is provider-agnostic.

**The three things to remember:**

1. **LLMs are stateless.** The buffer simulates memory by re-sending the full transcript on every call. The model reads it. That is the whole mechanism.

2. **Always set a token budget.** Without it, cost grows quadratically. A power user's Turn 10 can cost 50x Turn 1. This shows up on your OpenAI bill before it shows up in code review.

3. **Buffer memory is the foundation, not the solution.** Technique 02 fixes cost with a sliding window. Technique 03 compresses with summaries. Technique 06 adds cross-session persistence. Technique 24 (Graphiti) solves the stale-fact problem entirely. Every one of those techniques builds on what we built here.

